In [ ]:
# === CELL 2: Download transcript ===

import re
import datetime
import urllib.request
import json
import html as html_lib
import os

# Fråga efter URL (stöd för env var YOUTUBE_URL eller interaktiv input)
URL = os.environ.get('YOUTUBE_URL')
if not URL or URL.strip() == '':
    try:
        URL = input("YouTube URL: ").strip()
    except Exception:
        raise ValueError("❌ Ingen URL angiven! Sätt miljövariabeln YOUTUBE_URL eller kör interaktivt.")

if not URL:
    raise ValueError("❌ Ingen URL angiven!")

# Extrahera video-ID
vid_match = re.search(r"([a-zA-Z0-9_-]{11})", URL)
if not vid_match:
    raise ValueError("❌ Kunde inte hitta video-ID i URL:en")
vid = vid_match.group(1)
print(f"🔍 Video ID: {vid}")

# Hämta metadata från YouTube-sidan
print("📡 Hämtar metadata...")
page_html = urllib.request.urlopen(f"https://www.youtube.com/watch?v={vid}").read().decode()

# Extrahera titel
title_match = re.search(r"<title>(.+?) - YouTube</title>", page_html)
title_original = title_match.group(1) if title_match else "Unknown Title"
title_clean = re.sub(r"[^a-zA-Z0-9åäöÅÄÖ ]", "", title_original).replace(" ", "_")[:50]

# Extrahera kanal
channel_match = re.search(r'link itemprop="name" content="([^"]+)"', page_html)
channel_original = channel_match.group(1) if channel_match else "Unknown Channel"
channel_clean = re.sub(r"[^a-zA-Z0-9åäöÅÄÖ ]", "", channel_original).replace(" ", "_")[:30]

# Extrahera uppladdningsdatum
upload_date_match = re.search(r'"uploadDate":"(\d{4}-\d{2}-\d{2})"', page_html)
upload_date = upload_date_match.group(1) if upload_date_match else "Unknown"

# Extrahera beskrivning
description_match = re.search(r'"shortDescription":"(.*?)"(?:,"isCrawlable")', page_html)
if description_match:
    try:
        description = json.loads('"' + description_match.group(1) + '"')
    except (json.JSONDecodeError, ValueError):
        description = description_match.group(1)
else:
    description = "No description available"

print(f"📺 Titel: {title_original}")
print(f"👤 Kanal: {channel_original}")
print(f"📅 Datum: {upload_date}")

# Hämta transkript direkt från YouTubes caption-API
print("📜 Hämtar transkript...")
base_url_match = re.search(
    r'"baseUrl":"(https://www\.youtube\.com/api/timedtext[^"]+)"',
    page_html,
)
if not base_url_match:
    raise ValueError("❌ Inget transkript tillgängligt för denna video")

# Avkoda alla JSON-unicodeescapes i URL:en (t.ex. \u0026 → &)
base_url = re.sub(
    r'\\u([0-9a-fA-F]{4})',
    lambda m: chr(int(m.group(1), 16)),
    base_url_match.group(1)
)

# Begär JSON3-format
base_url = re.sub(r'fmt=[^&]*', 'fmt=json3', base_url)
if "fmt=" not in base_url:
    base_url += "&fmt=json3"

req = urllib.request.Request(
    base_url,
    headers={"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"}
)
data = json.loads(urllib.request.urlopen(req).read().decode())

texts = []
for event in data.get("events", []):
    for seg in event.get("segs", []):
        text = seg.get("utf8", "").replace("\n", " ").strip()
        if text:
            texts.append(text)
transcript_text = " ".join(texts)
print(f"📏 Längd: {len(transcript_text):,} tecken")

# Skapa header
now = datetime.datetime.now().strftime("%Y%m%d_%H%M")
header = f"""This file is a transcript from YouTube
Channel: {channel_original}
Title: {title_original}
Upload date: {upload_date}
URL: https://www.youtube.com/watch?v={vid}

Description:
{description}

---

"""

# Spara till fil
output_folder = "transcripts"
os.makedirs(output_folder, exist_ok=True)

filename = f"{now}_{channel_clean}_-_{title_clean}.txt"
filepath = os.path.join(output_folder, filename)

with open(filepath, "w", encoding="utf-8") as f:
    f.write(header + transcript_text)

print(f"\n✅ Transkript sparat!")
print(f"📁 Fil: {filepath}")

# Spara variabler för nästa cell
TRANSCRIPT_CONTENT = header + transcript_text
FILEPATH = filepath
TITLE = title_original
VID = vid

In [ ]:
# === CELL 1: Install packages (run once) ===
!pip install youtube_transcript_api python-dotenv